# TP1 â€“ Continual Learning on Seq-CIFAR-10
**I309 â€“ VisiÃ³n Artificial Avanzada Â· Universidad de San AndrÃ©s**

This notebook is the single entry point for all experiments.
It follows the four stages of the assignment:

1. Dataset preparation (Seq-CIFAR-10, replay buffer)
2. Pre-training with Supervised Contrastive Learning (SupCon)
3. Continual Learning methods: Naive, EWC, LwF, CoÂ²L
4. Comparison of results (Class-IL and Task-IL)


## 0. Setup

In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # avoid OpenMP conflict on Windows

import sys, os

# __file__ doesn't exist in Jupyter â€” use the working directory instead.
# Launch Jupyter from the repo root (the folder containing tp1.ipynb) and
# this will resolve correctly.
REPO_ROOT = os.path.abspath('')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'REPO_ROOT: {REPO_ROOT}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {DEVICE}')

PyTorch  : 2.2.2+cpu
REPO_ROOT: c:\Users\Hp\OneDrive\Documentos\UdeSA\5_ano\Vision_artificial_avanzada\TP1\Continual-Learning-on-Seq-CIFAR-10
Device   : cpu


In [2]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

from data.dataset import SeqCIFAR10, TASK_CLASSES, CLASS_NAMES
from data.buffer import ReplayBuffer
from models.backbone import Backbone
from losses.supcon import SupConLoss
from losses.distillation import AsymmetricDistillationLoss
from methods.naive import NaiveFineTuning
from methods.ewc import EWC
from methods.lwf import LwF
from methods.co2l import Co2L
from methods.trainer import ContinualTrainer
from methods.pretrain import pretrain_supcon, train_linear_probe, load_pretrained_backbone
from utils.metrics import evaluate_class_il, evaluate_task_il, compute_forgetting, MetricsTracker
from utils.visualization import (plot_accuracy_curve, plot_forgetting_curve,
    plot_embeddings, plot_loss_curve,
    plot_pretrain_loss, plot_embedding_stages, plot_probe_accuracy)

print('All imports OK')

All imports OK


## Stage 1 â€“ Dataset Preparation

Split CIFAR-10 into 5 sequential tasks (2 classes each) and set up the replay buffer.

| Task | Classes               | Train images |
|------|-----------------------|--------------|
| 0    | airplane, automobile  | 10 000       |
| 1    | bird, cat             | 10 000       |
| 2    | deer, dog             | 10 000       |
| 3    | frog, horse           | 10 000       |
| 4    | ship, truck           | 10 000       |

In [3]:
DATA_ROOT   = './data'   # torchvision downloads CIFAR-10 here on first run
N_TASKS     = 5
BATCH_SIZE  = 128
BUFFER_SIZE = 200        # replay buffer capacity across all past tasks

seq_cifar = SeqCIFAR10(data_root=DATA_ROOT, n_tasks=N_TASKS, batch_size=BATCH_SIZE)
buffer    = ReplayBuffer(capacity=BUFFER_SIZE)

print(f'Task splits (class indices): {TASK_CLASSES}')
print()
for t in range(N_TASKS):
    names = seq_cifar.get_class_names(t)
    n_train = seq_cifar.task_size(t, train=True)
    n_test  = seq_cifar.task_size(t, train=False)
    print(f'  Task {t}: {names[0]:12s} + {names[1]:12s}  |  train={n_train:5d}  test={n_test:4d}')

100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 170498071/170498071 [00:37<00:00, 4535451.23it/s]


Extracting ./data\cifar-10-python.tar.gz to ./data
Files already downloaded and verified
Task splits (class indices): [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

  Task 0: airplane     + automobile    |  train=10000  test=2000
  Task 1: bird         + cat           |  train=10000  test=2000
  Task 2: deer         + dog           |  train=10000  test=2000
  Task 3: frog         + horse         |  train=10000  test=2000
  Task 4: ship         + truck         |  train=10000  test=2000


In [5]:
# --- Verify class filtering is correct ---
print('Verifying class filtering...')
for t in range(N_TASKS):
    c0, c1 = seq_cifar.get_classes(t)
    loader = seq_cifar.get_task_loader(t, train=True)
    seen_classes = set()
    for _, labels in loader:
        seen_classes.update(labels.tolist())
    assert seen_classes == {c0, c1}, f'Task {t}: expected {{{c0},{c1}}}, got {seen_classes}'
    print(f'  Task {t} OK â€” classes {seen_classes} = {seq_cifar.get_class_names(t)}')

print()
# Verify a SupCon loader returns (B, 2, C, H, W)
supcon_loader = seq_cifar.get_task_loader(0, train=True, supcon=True)
imgs, labels = next(iter(supcon_loader))
assert imgs.ndim == 5 and imgs.shape[1] == 2 and imgs.shape[2:] == (3, 32, 32), \
    f'Expected (B,2,3,32,32) but got {imgs.shape}'
print(f'SupCon loader batch shape: {imgs.shape}  â†’  (B={imgs.shape[0]}, 2 views, C=3, H=32, W=32)')
print()
print('All checks passed.')

Verifying class filtering...
  Task 0 OK â€” classes {0, 1} = ('airplane', 'automobile')
  Task 1 OK â€” classes {2, 3} = ('bird', 'cat')
  Task 2 OK â€” classes {4, 5} = ('deer', 'dog')
  Task 3 OK â€” classes {6, 7} = ('frog', 'horse')
  Task 4 OK â€” classes {8, 9} = ('ship', 'truck')

SupCon loader batch shape: torch.Size([128, 2, 3, 32, 32])  â†’  (B=128, 2 views, C=3, H=32, W=32)

All checks passed.


## Stage 2 â€“ Pre-training with Supervised Contrastive Learning

Train the backbone encoder on Task 0 using SupCon loss, then attach and train a linear classification head.

In [6]:
# ── Backbone ──────────────────────────────────────────────────────────────────
backbone_pretrained = Backbone(num_classes=10).to(DEVICE)

# ── SupCon pre-training on Task 0 (airplane / automobile) ─────────────────────
from methods.pretrain import pretrain_supcon, load_pretrained_backbone

SUPCON_EPOCHS    = 200        # set lower (e.g. 20) for a quick smoke-test
SUPCON_LR        = 0.5
SUPCON_TEMP      = 0.07
CHECKPOINT_DIR   = "checkpoints"
CHECKPOINT_EVERY = 50         # save intermediate ckpts every 50 epochs (0 = off)

pretrain_history = pretrain_supcon(
    backbone        = backbone_pretrained,
    seq_cifar       = seq_cifar,
    task_id         = 0,
    n_epochs        = SUPCON_EPOCHS,
    lr              = SUPCON_LR,
    temperature     = SUPCON_TEMP,
    device          = DEVICE,
    checkpoint_dir  = CHECKPOINT_DIR,
    checkpoint_every= CHECKPOINT_EVERY,
    save_stages     = True,
)

print(f"Pre-training done.  Final loss: {pretrain_history['loss_history'][-1]:.4f}")
print(f"Checkpoint saved to: {pretrain_history['checkpoint_path']}")


Backbone(
  encoder  : ResNet-18 for 32Ã—32  [trainable]
  feat_dim : 512
  projector: Linear(512, 512) â†’ ReLU â†’ Linear(512, 128) â†’ L2-norm
  classifier: Linear(512, 10)
)
Shape checks OK
Encoder frozen: False


In [7]:
# ── SupCon loss curve ─────────────────────────────────────────────────────────
from utils.visualization import plot_pretrain_loss

plot_pretrain_loss(
    loss_history = pretrain_history["loss_history"],
    save_path    = "imgs/supcon_pretrain_loss.png",
    title        = "SupCon Pre-training Loss — Task 0 (airplane / automobile)",
)
print("Saved: imgs/supcon_pretrain_loss.png")


In [ ]:
# ── Embedding projections: init / mid / final ─────────────────────────────────
# Loads backbone snapshots saved during pre-training and runs t-SNE on
# Task 0 test features (airplane / automobile), producing a 1x3 side-by-side.
from utils.visualization import plot_embedding_stages

test_loader_viz = seq_cifar.get_task_loader(0, train=False)

plot_embedding_stages(
    stage_paths       = pretrain_history["stage_paths"],
    backbone_template = backbone_pretrained,
    loader            = test_loader_viz,
    device            = DEVICE,
    class_names       = CLASS_NAMES,
    save_path         = "imgs/supcon_embeddings_stages.png",
    n_samples         = 1000,
    method            = "tsne",
)
print("Saved: imgs/supcon_embeddings_stages.png")


In [ ]:
# ── Linear probe on Task 0 ────────────────────────────────────────────────────
# Encoder is frozen inside train_linear_probe(); the function verifies this
# every epoch and raises RuntimeError if the freeze is ever violated.

LINEAR_EPOCHS = 100
LINEAR_LR     = 1.0

probe_history = train_linear_probe(
    backbone       = backbone_pretrained,
    seq_cifar      = seq_cifar,
    task_id        = 0,
    n_epochs       = LINEAR_EPOCHS,
    lr             = LINEAR_LR,
    device         = DEVICE,
    checkpoint_dir = CHECKPOINT_DIR,
)

print(f"Encoder frozen after probe: {backbone_pretrained.is_encoder_frozen}")
print(f"Linear probe — Task 0 test accuracy : {probe_history['test_acc']:.3f}")
print(f"Checkpoint: {probe_history['checkpoint_path']}")


In [ ]:
# ── Linear probe results ──────────────────────────────────────────────────────
from utils.visualization import plot_probe_accuracy

plot_probe_accuracy(
    probe_history = probe_history,
    save_path     = "imgs/linear_probe_curves.png",
    title         = "Linear Probe — Task 0",
)
print(f"Test accuracy: {probe_history['test_acc']:.3f}")
print("Saved: imgs/linear_probe_curves.png")


In [8]:
# --- 3.1 Naive Fine-Tuning ------------------------------------------------
naive_tracker  = MetricsTracker(n_tasks=N_TASKS)
naive_backbone = copy.deepcopy(backbone)
naive          = NaiveFineTuning(naive_backbone, device=DEVICE, lr=0.01)
naive_trainer  = ContinualTrainer(
    naive, seq_cifar, n_epochs=50,
    eval_fn=make_eval_fn(seq_cifar, DEVICE, naive_tracker),
)
naive_results = naive_trainer.train_all_tasks()
print(naive_tracker)
naive_tracker.summary_df()

INFO [ContinualTrainer] task=0  method=NaiveFineTuning  loader_classes=('airplane', 'automobile')  replay=False
INFO [NaiveFineTuning] begin_task  task=0  classes={0, 1}  replay=False



  ContinualTrainer â€” NaiveFineTuning
  Tasks: 5   Epochs/task: 50
  Replay: False

[Task 0] airplane + automobile  (10000 train samples)
  Data source : get_task_loader(task_id=0, train=True)
  Past data   : none


INFO [NaiveFineTuning] task=0  epoch=1/50  loss=0.5066  acc=0.756
INFO [NaiveFineTuning] task=0  epoch=2/50  loss=0.3064  acc=0.870
INFO [NaiveFineTuning] task=0  epoch=3/50  loss=0.2359  acc=0.905
INFO [NaiveFineTuning] task=0  epoch=4/50  loss=0.2186  acc=0.915
INFO [NaiveFineTuning] task=0  epoch=5/50  loss=0.1711  acc=0.932
INFO [NaiveFineTuning] task=0  epoch=6/50  loss=0.1544  acc=0.940
INFO [NaiveFineTuning] task=0  epoch=7/50  loss=0.1521  acc=0.940
INFO [NaiveFineTuning] task=0  epoch=8/50  loss=0.1154  acc=0.956
INFO [NaiveFineTuning] task=0  epoch=9/50  loss=0.1031  acc=0.959
INFO [NaiveFineTuning] task=0  epoch=10/50  loss=0.0830  acc=0.969
INFO [NaiveFineTuning] task=0  epoch=11/50  loss=0.0870  acc=0.968
INFO [NaiveFineTuning] task=0  epoch=12/50  loss=0.0758  acc=0.971
INFO [NaiveFineTuning] task=0  epoch=13/50  loss=0.0617  acc=0.977
INFO [NaiveFineTuning] task=0  epoch=14/50  loss=0.0897  acc=0.968
INFO [NaiveFineTuning] task=0  epoch=15/50  loss=0.0545  acc=0.981
INFO

  Done in 65277.1s â€” final loss=0.0096  acc=0.997


INFO [ContinualTrainer] task=1  method=NaiveFineTuning  loader_classes=('bird', 'cat')  replay=False
INFO [NaiveFineTuning] begin_task  task=1  classes={2, 3}  replay=False


  Eval after task 0: class_il=0.984  task_il=0.984

[Task 1] bird + cat  (10000 train samples)
  Data source : get_task_loader(task_id=1, train=True)
  Past data   : none


INFO [NaiveFineTuning] task=1  epoch=1/50  loss=1.2896  acc=0.686
INFO [NaiveFineTuning] task=1  epoch=2/50  loss=0.4276  acc=0.806
INFO [NaiveFineTuning] task=1  epoch=3/50  loss=0.3737  acc=0.839
INFO [NaiveFineTuning] task=1  epoch=4/50  loss=0.3592  acc=0.851


KeyboardInterrupt: 

In [ ]:
# --- 3.2 EWC --------------------------------------------------------------
# Protocol: no replay â€” penalty term on Fisher matrix prevents forgetting.
# TODO: uncomment once EWC.train_task / end_task are implemented.

# ewc_backbone = copy.deepcopy(backbone)
# ewc = EWC(ewc_backbone, device=DEVICE, ewc_lambda=400.0)
# ewc_trainer = ContinualTrainer(ewc, seq_cifar, n_epochs=50, eval_fn=eval_fn)
# ewc_results = ewc_trainer.train_all_tasks()
# ewc_class_il = [ewc_results["eval_results"][t]["class_il"] for t in range(N_TASKS)]
# ewc_task_il  = [ewc_results["eval_results"][t]["task_il"]  for t in range(N_TASKS)]

In [ ]:
# --- 3.3 LwF --------------------------------------------------------------
# Protocol: no replay â€” knowledge distillation from the previous model.
# TODO: uncomment once LwF.train_task is implemented.

# lwf_backbone = copy.deepcopy(backbone)
# lwf = LwF(lwf_backbone, device=DEVICE)
# lwf_trainer = ContinualTrainer(lwf, seq_cifar, n_epochs=50, eval_fn=eval_fn)
# lwf_results = lwf_trainer.train_all_tasks()
# lwf_class_il = [lwf_results["eval_results"][t]["class_il"] for t in range(N_TASKS)]
# lwf_task_il  = [lwf_results["eval_results"][t]["task_il"]  for t in range(N_TASKS)]

In [ ]:
# --- 3.4 CoÂ²L -------------------------------------------------------------
# Protocol: replay-enabled â€” SupCon on current-task data + buffer samples.
# uses_replay=True is declared on Co2L; ContinualTrainer logs this per task.
# TODO: uncomment once Co2L.train_task / end_task are implemented.

# co2l_backbone = copy.deepcopy(backbone)
# co2l_buffer   = ReplayBuffer(capacity=BUFFER_SIZE)
# co2l = Co2L(co2l_backbone, device=DEVICE, replay_buffer=co2l_buffer)
# co2l_trainer = ContinualTrainer(co2l, seq_cifar, n_epochs=500, eval_fn=eval_fn)
# co2l_results = co2l_trainer.train_all_tasks()
# co2l_class_il = [co2l_results["eval_results"][t]["class_il"] for t in range(N_TASKS)]
# co2l_task_il  = [co2l_results["eval_results"][t]["task_il"]  for t in range(N_TASKS)]

## Stage 4 â€“ Comparison of Results

In [ ]:
# --- Stage 4: Comparison of Results --------------------------------------
# Collect avg-accuracy curves from each tracker (uncomment as methods are done)
all_trackers = {
    "Naive": naive_tracker,
    # "EWC":   ewc_tracker,
    # "LwF":   lwf_tracker,
    # "Co2L":  co2l_tracker,
}

# Accuracy curves: avg Class-IL / Task-IL vs. number of tasks learned
plot_accuracy_curve(
    {m: t.avg_acc_curve("class_il") for m, t in all_trackers.items()},
    scenario="Class-IL",
)
plot_accuracy_curve(
    {m: t.avg_acc_curve("task_il") for m, t in all_trackers.items()},
    scenario="Task-IL",
)

# Forgetting curves
plot_forgetting_curve(
    {m: t.forgetting("class_il") for m, t in all_trackers.items()},
)

In [ ]:
# Final summary table: one row per method, final Class-IL / Task-IL / forgetting
import pandas as pd

rows = []
for method_name, tracker in all_trackers.items():
    rows.append({
        "Method":         method_name,
        "Class-IL":       round(tracker.avg_acc_curve("class_il")[-1], 4),
        "Task-IL":        round(tracker.avg_acc_curve("task_il")[-1],  4),
        "Avg Forgetting": round(tracker.avg_forgetting("class_il"),    4),
    })

pd.DataFrame(rows).set_index("Method")